# Checkpoint 7 — refined subscriber-tier-only model

This experiment removes raw subscriber count from the model and retains only fixed subscriber tiers. The former 100k–1m tier is divided into 100k–250k, 250k–500k, and 500k–1m.

The horizon rows, channel-grouped folds, XGBoost parameters, target transformation, and reserved test remain unchanged, making this a controlled comparison.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_checkpoint5_models import (
    HORIZONS,
    MODEL_NAME,
    load_horizon_checkpoint,
    train_all_horizons,
)
from viewcastlk_ml.horizon_preprocessing import (
    SUBSCRIBER_TIER_ORDER,
    HorizonDatasetPreprocessor,
)

RAW_MODEL_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint5_xgboost'
RAW_COARSE_TIER_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint6_subscriber_tier'
TIER_ONLY_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint7_refined_tier_only'

loaded = {}
tier_rows = []
for horizon in HORIZONS:
    X, y, assignments, _, _ = load_horizon_checkpoint(PROJECT_ROOT, horizon)
    loaded[horizon] = (X, y, assignments)
    assert 'ch_subs_at_publish' not in X.columns
    assert 'subscriber_tier' in X.columns
    assert X['subscriber_tier'].notna().all()
    counts = X['subscriber_tier'].value_counts().reindex(SUBSCRIBER_TIER_ORDER, fill_value=0)
    for tier, count in counts.items():
        tier_rows.append({
            'horizon_days': horizon,
            'subscriber_tier': tier,
            'video_rows': int(count),
            'percent': 100 * count / len(X),
        })

tier_distribution = pd.DataFrame(tier_rows)
print('Refined tier percentage by horizon')
display(tier_distribution.pivot(
    index='subscriber_tier', columns='horizon_days', values='percent'
).reindex(SUBSCRIBER_TIER_ORDER).round(2))

Refined tier percentage by horizon
horizon_days        7      14     21     30
subscriber_tier                            
under_1k          6.91   7.22   7.66   7.73
1k_to_10k        15.21  17.68  20.26  19.03
10k_to_100k      16.40  19.54  20.75  21.40
100k_to_250k      6.09   6.38   6.95   6.98
250k_to_500k      4.36   5.41   5.13   5.26
500k_to_1m       10.81   9.30   8.95   9.17
1m_plus          40.23  34.48  30.29  30.43
missing           0.00   0.00   0.00   0.00


In [2]:
# Show the actual tier-only model matrix on a validation sample.
X7, y7, assignments7 = loaded[7]
training_mask = assignments7['partition'].eq('development') & ~assignments7['cv_validation_fold'].eq(1)
validation_mask = assignments7['partition'].eq('development') & assignments7['cv_validation_fold'].eq(1)
training_positions = assignments7.loc[training_mask, 'horizon_row_position'].astype(int).to_numpy()
validation_positions = assignments7.loc[validation_mask, 'horizon_row_position'].astype(int).to_numpy()

preprocessor = HorizonDatasetPreprocessor()
preprocessor.fit(X7.iloc[training_positions], np.log1p(y7.iloc[training_positions]))
source_preview = X7.iloc[validation_positions[:8]]
model_preview = preprocessor.transform(source_preview)
tier_features = [column for column in model_preview if column.startswith('subscriber_tier_')]

print('Tier model features:', tier_features)
display(pd.concat([
    source_preview[['subscriber_tier']].reset_index(drop=True),
    model_preview[tier_features].reset_index(drop=True),
], axis=1))

assert 'ch_subs_at_publish' not in source_preview.columns
assert 'ch_subs_at_publish' not in model_preview.columns
assert len(tier_features) == 7
assert np.allclose(model_preview[tier_features].sum(axis=1), 1.0)
print('PASS: raw subscriber count is absent and exactly one refined tier is active per row.')

Tier model features: ['subscriber_tier_100k_to_250k', 'subscriber_tier_10k_to_100k', 'subscriber_tier_1k_to_10k', 'subscriber_tier_1m_plus', 'subscriber_tier_250k_to_500k', 'subscriber_tier_500k_to_1m', 'subscriber_tier_under_1k']
  subscriber_tier  ...  subscriber_tier_under_1k
0         1m_plus  ...                       0.0
1         1m_plus  ...                       0.0
2         1m_plus  ...                       0.0
3       1k_to_10k  ...                       0.0
4         1m_plus  ...                       0.0
5         1m_plus  ...                       0.0
6      500k_to_1m  ...                       0.0
7         1m_plus  ...                       0.0

[8 rows x 8 columns]
PASS: raw subscriber count is absent and exactly one refined tier is active per row.


In [3]:
training_run = train_all_horizons(
    project_root=PROJECT_ROOT,
    output_dir=TIER_ONLY_DIR,
    n_estimators=800,
    n_jobs=4,
    include_llm_scores=False,
)

display(training_run['summary'][[
    'horizon_days', 'model', 'rows', 'mape_nonzero_pct',
    'median_ape_nonzero_pct', 'smape_pct', 'rmsle', 'log_r2'
]])


Training independent day-7 model
day 7 fold 1/5: RMSLE=2.0237, median APE=95.03%, best trees=13
day 7 fold 2/5: RMSLE=2.1988, median APE=86.52%, best trees=27
day 7 fold 3/5: RMSLE=2.1246, median APE=92.30%, best trees=190
day 7 fold 4/5: RMSLE=1.9224, median APE=86.09%, best trees=67
day 7 fold 5/5: RMSLE=2.4407, median APE=150.46%, best trees=12

Training independent day-14 model
day 14 fold 1/5: RMSLE=1.8719, median APE=97.68%, best trees=164
day 14 fold 2/5: RMSLE=2.2047, median APE=90.42%, best trees=2
day 14 fold 3/5: RMSLE=2.5132, median APE=98.11%, best trees=318
day 14 fold 4/5: RMSLE=1.9300, median APE=88.56%, best trees=94
day 14 fold 5/5: RMSLE=2.1537, median APE=95.16%, best trees=279

Training independent day-21 model
day 21 fold 1/5: RMSLE=1.9749, median APE=119.88%, best trees=29
day 21 fold 2/5: RMSLE=1.9443, median APE=86.43%, best trees=105
day 21 fold 3/5: RMSLE=2.0303, median APE=89.98%, best trees=104
day 21 fold 4/5: RMSLE=1.9893, median APE=91.38%, best trees=3

In [4]:
# Compare all three subscriber representations.
runs = {
    'raw_subscriber': pd.read_csv(RAW_MODEL_DIR / 'cv_summary_metrics.csv', dtype={'horizon_days': str}),
    'raw_plus_coarse_tier': pd.read_csv(RAW_COARSE_TIER_DIR / 'cv_summary_metrics.csv', dtype={'horizon_days': str}),
    'refined_tier_only': pd.read_csv(TIER_ONLY_DIR / 'cv_summary_metrics.csv', dtype={'horizon_days': str}),
}
comparison_rows = []
for horizon in [str(h) for h in HORIZONS] + ['combined']:
    for run_name, summary in runs.items():
        result = summary[
            summary['horizon_days'].eq(horizon)
            & summary['model'].eq(MODEL_NAME)
        ].iloc[0]
        comparison_rows.append({
            'horizon_days': horizon,
            'subscriber_representation': run_name,
            'rmsle': result['rmsle'],
            'mape_pct': result['mape_nonzero_pct'],
            'median_ape_pct': result['median_ape_nonzero_pct'],
            'smape_pct': result['smape_pct'],
            'log_r2': result['log_r2'],
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))

combined_comparison = comparison[comparison['horizon_days'].eq('combined')].set_index('subscriber_representation')
display(combined_comparison.round(4))

   horizon_days subscriber_representation  ...  smape_pct  log_r2
0             7            raw_subscriber  ...   117.0905  0.3005
1             7      raw_plus_coarse_tier  ...   116.3905  0.3352
2             7         refined_tier_only  ...   117.5701  0.2470
3            14            raw_subscriber  ...   117.1304  0.2991
4            14      raw_plus_coarse_tier  ...   117.6134  0.3043
5            14         refined_tier_only  ...   118.1228  0.2775
6            21            raw_subscriber  ...   114.0420  0.3665
7            21      raw_plus_coarse_tier  ...   114.8247  0.3564
8            21         refined_tier_only  ...   116.3217  0.3250
9            30            raw_subscriber  ...   114.5122  0.3753
10           30      raw_plus_coarse_tier  ...   114.3245  0.3825
11           30         refined_tier_only  ...   115.2575  0.3801
12     combined            raw_subscriber  ...   115.8307  0.3329
13     combined      raw_plus_coarse_tier  ...   115.8630  0.3439
14     com

In [5]:
# Saved-artifact and leakage tests.
manifest = json.loads((TIER_ONLY_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
predictions = pd.read_csv(TIER_ONLY_DIR / 'cv_predictions.csv')
reference_predictions = pd.read_csv(RAW_MODEL_DIR / 'cv_predictions.csv')
test_rows = []

def check(name, condition, detail=''):
    test_rows.append({'test': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail})

check('raw subscriber explicitly excluded', 'ch_subs_at_publish' in manifest['explicitly_excluded_model_columns'])
check('reserved test remains unevaluated', manifest['status'] == 'candidate_reserved_test_not_evaluated')
check('same OOF rows as raw-subscriber model', set(zip(predictions['horizon_days'], predictions['horizon_row_position'])) == set(zip(reference_predictions['horizon_days'], reference_predictions['horizon_row_position'])))
check('predictions finite', np.isfinite(predictions.filter(like='predicted_').to_numpy(dtype=float)).all())

for record in manifest['models']:
    horizon = record['horizon_days']
    bundle = joblib.load(TIER_ONLY_DIR / record['model_path'])
    feature_order = bundle.feature_names
    tiers = [feature for feature in feature_order if feature.startswith('subscriber_tier_')]
    X, y, assignments = loaded[horizon]
    development_positions = set(assignments.loc[assignments['partition'].eq('development'), 'horizon_row_position'].astype(int))
    reserved_positions = set(assignments.loc[assignments['partition'].eq('test_reserved'), 'horizon_row_position'].astype(int))
    predicted_positions = set(predictions.loc[predictions['horizon_days'].eq(horizon), 'horizon_row_position'].astype(int))
    sample_prediction = bundle.predict_views(X.iloc[[min(development_positions)]])

    check(f'day {horizon} raw subscriber absent from model', 'ch_subs_at_publish' not in feature_order)
    check(f'day {horizon} seven refined tier indicators saved', len(tiers) == 7, ', '.join(tiers))
    check(f'day {horizon} development coverage exact', predicted_positions == development_positions)
    check(f'day {horizon} reserved rows absent', predicted_positions.isdisjoint(reserved_positions))
    check(f'day {horizon} bundle reloads and predicts', len(sample_prediction) == 1 and np.isfinite(sample_prediction).all())

tests = pd.DataFrame(test_rows)
display(tests)
failures = tests[tests['status'].eq('FAIL')]
assert failures.empty, failures.to_string(index=False)
print(f'PASS: all {len(tests)} refined-tier-only checks succeeded.')

                                          test  ...                                             detail
0           raw subscriber explicitly excluded  ...                                                   
1            reserved test remains unevaluated  ...                                                   
2        same OOF rows as raw-subscriber model  ...                                                   
3                           predictions finite  ...                                                   
4       day 7 raw subscriber absent from model  ...                                                   
5    day 7 seven refined tier indicators saved  ...  subscriber_tier_100k_to_250k, subscriber_tier_...
6             day 7 development coverage exact  ...                                                   
7                   day 7 reserved rows absent  ...                                                   
8            day 7 bundle reloads and predicts  ...                      

## Checkpoint decision

The comparison table determines whether the refined tier-only representation performs better than raw subscriber count or raw count plus the earlier coarse tier. The reserved test remains untouched.